# BEAM Benchmark — UMD JupyterHub (GPU Cluster)

Runs the BEAM benchmark using **Gemma 4 31B from the UMD GPU cluster** (gpu02 vLLM) for LLM calls and **Fireworks.ai** for embeddings only.

## Architecture
- **LLM (chat/extraction/QA):** `google/gemma-4-31B-it` via vLLM on gpu02 — **no rate limits**
- **Embeddings:** `qwen3-embedding-8b` via Fireworks API — small requests, minimal rate-limit risk
- **Judge (Phase 2):** Local Gemma by default (free, no rate limits)
- **Execution:** UMD JupyterHub (gpuyter.mind.cs.umd.edu) — persists when laptop is down
- **Storage:** GlusterFS home dir (`~/beam_results/`) — survives session disconnects

## Prerequisites
1. **Push your code changes to GitHub** before running this notebook (the notebook clones from GitHub)
2. Log in to [gpuyter.mind.cs.umd.edu](https://gpuyter.mind.cs.umd.edu) and open this notebook
3. **Switch kernel to Python 3.11 (beam)** — Kernel → Change kernel → Python 3.11 (beam)
4. Run cells top-to-bottom

## Network setup (do once in a JupyterHub terminal)
The JupyterHub container cannot reach gpu02:8000 directly. You need an SSH tunnel:

```bash
# In a JupyterHub terminal (File → New → Terminal):
ssh -L 8000:localhost:8000 nmokaria@gpu02.mind.cs.umd.edu -N &
```

Also start vLLM on gpu02 (if not already running):

```bash
ssh nmokaria@gpu02.mind.cs.umd.edu
nohup env TRITON_CACHE_DIR=/scratch/triton_cache_gemma VLLM_USE_FLASHINFER_SAMPLER=0 CUDA_VISIBLE_DEVICES=0,1 \
  python -m vllm.entrypoints.openai.api_server \
  --model google/gemma-4-31B-it \
  --tensor-parallel-size 2 --port 8000 --host 0.0.0.0 \
  --max-model-len 32768 --max-num-batched-tokens 4096 \
  --gpu-memory-utilization 0.85 \
  > /scratch/vllm_gemma.log 2>&1 &
```

## Notes
- vLLM is accessed via SSH tunnel at `localhost:8000` (not directly at gpu02)
- If JupyterHub disconnects, re-run with `--resume` to continue from checkpoints
- The SSH tunnel must be re-established if the JupyterHub session restarts

## 1. Clone repo & install dependencies

In [ ]:
import os, sys

REPO_DIR = os.path.expanduser('~/Agentic-Graph-Memory')

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/nmokaria27/Agentic-Graph-Memory.git {REPO_DIR}

%cd {REPO_DIR}

# Checkout the feature branch
!git checkout feat/vector-index-and-qa-improvements
!git pull origin feat/vector-index-and-qa-improvements

# Install dependencies using the beam env's Python (not system Python 3.7)
BEAM_PY = '/opt/conda/envs/beam/bin/python'
!{BEAM_PY} -m pip install --no-build-isolation -e . -q
!{BEAM_PY} -m pip install datasets json_repair sentence_transformers scipy -q

print(f'\n=== Dependencies installed ===')
print(f'Python: {sys.version}')
print(f'Executable: {sys.executable}')

## 2. Verify vLLM is running (via SSH tunnel)

Make sure you've done these in a **JupyterHub terminal** (File → New → Terminal):

**Terminal 1 — SSH tunnel:**
```bash
ssh -L 8000:localhost:8000 nmokaria@gpu02.mind.cs.umd.edu -N &
```

**Terminal 2 — Start vLLM on gpu02 (if not already running):**
```bash
ssh nmokaria@gpu02.mind.cs.umd.edu
nohup env TRITON_CACHE_DIR=/scratch/triton_cache_gemma VLLM_USE_FLASHINFER_SAMPLER=0 CUDA_VISIBLE_DEVICES=0,1 \
  python -m vllm.entrypoints.openai.api_server \
  --model google/gemma-4-31B-it \
  --tensor-parallel-size 2 --port 8000 --host 0.0.0.0 \
  --max-model-len 32768 --max-num-batched-tokens 4096 \
  --gpu-memory-utilization 0.85 \
  > /scratch/vllm_gemma.log 2>&1 &
```

Wait ~2-3 minutes, then run the cell below to confirm vLLM is reachable via the tunnel.

In [ ]:
import requests, time

# vLLM is accessed via SSH tunnel at 127.0.0.1:8000 (use 127.0.0.1, NOT localhost —
# localhost resolves to ::1 (IPv6) which the SSH tunnel doesn't bind to)
VLLM_HOST = '127.0.0.1'
VLLM_PORT = 8000
VLLM_URL = f'http://{VLLM_HOST}:{VLLM_PORT}'

# Poll until vLLM is ready (check every 10s, up to 5 minutes)
print(f'Checking vLLM at {VLLM_URL} (via SSH tunnel to gpu02) ...')
for i in range(30):
    try:
        resp = requests.get(f'{VLLM_URL}/v1/models', timeout=5)
        if resp.status_code == 200:
            models = resp.json().get('data', [])
            print(f'✅ vLLM ready — {len(models)} model(s) loaded:')
            for m in models:
                print(f'   {m["id"]}')
            break
    except Exception:
        pass
    print(f'  [{(i+1)*10}s] not ready yet — waiting...', end='\r')
    time.sleep(10)
else:
    print('\n❌ vLLM not reachable after 5 minutes.')
    print('   → Open a JupyterHub terminal and run:')
    print('       ssh -L 127.0.0.1:8000:127.0.0.1:8000 nmokaria@gpu02.mind.cs.umd.edu -N')
    print('   → Then re-run this cell')

## 3. Configure `.env`

Sets up the environment to use:
- **Gemma 4 31B** from gpu02 vLLM for all LLM calls (no rate limits)
- **Fireworks** for embeddings only (qwen3-embedding-8b)
- **Local Gemma** as the judge model for Phase 2 scoring

You'll need your Fireworks API key. It will be read from `~/.fireworks_api_key` if available, or you'll be prompted to enter it.

In [ ]:
import os, getpass, stat

# ── Get Fireworks API key ──────────────────────────────────────────
key_file = os.path.expanduser('~/.fireworks_api_key')
FIREWORKS_API_KEY = None

if os.path.exists(key_file):
    with open(key_file) as f:
        FIREWORKS_API_KEY = f.read().strip()
    print(f'✅ Fireworks API key loaded from {key_file}')

if not FIREWORKS_API_KEY:
    FIREWORKS_API_KEY = getpass.getpass('Enter Fireworks API key: ')
    # Save for future sessions (restrict permissions)
    with open(key_file, 'w') as f:
        f.write(FIREWORKS_API_KEY)
    os.chmod(key_file, stat.S_IRUSR | stat.S_IWUSR)
    print(f'✅ Key saved to {key_file} (chmod 600)')

# ── Write .env ─────────────────────────────────────────────────────
env = f"""# ── Hybrid: Gemma (gpu02 vLLM) + Fireworks embeddings ────────────
LLM_BACKEND=vllm
VLLM_BASE_URL=http://127.0.0.1:8000/v1
VLLM_API_KEY=EMPTY
LLM_DEFAULT_MODEL=google/gemma-4-31B-it

# Embeddings via Fireworks (small requests, minimal rate-limit risk)
EMBEDDING_BASE_URL=https://api.fireworks.ai/inference/v1
EMBEDDING_API_KEY={FIREWORKS_API_KEY}
EMBEDDING_MODEL=accounts/fireworks/models/qwen3-embedding-8b

# ── Concurrency (local GPU — no rate limits, can be aggressive) ────
ENTITY_EXTRACT_WORKERS=6
RELATION_EXTRACT_WORKERS=6
LLM_RETRY_BACKOFF=2

# ── Judge model for Phase 2 scoring ───────────────────────────────
# Using Fireworks deepseek-v4-pro for higher-quality judging
BEAM_JUDGE_MODEL=accounts/fireworks/models/deepseek-v4-pro
"""

with open('.env', 'w') as f:
    f.write(env)

print('=== .env written ===')
print(f'   LLM:          google/gemma-4-31B-it via 127.0.0.1:8000')
print(f'   Embeddings:   qwen3-embedding-8b via Fireworks')
print(f'   Judge:        accounts/fireworks/models/deepseek-v4-pro (Fireworks)')

## 4. Verify connectivity

Check that both vLLM (gpu02) and Fireworks API are reachable from JupyterHub.

In [ ]:
import requests

# Check vLLM on gpu02
print('── vLLM (gpu02) ──')
try:
    resp = requests.get(f'{VLLM_URL}/v1/models', timeout=10)
    if resp.status_code == 200:
        models = resp.json().get('data', [])
        print(f'✅ vLLM reachable — {len(models)} model(s)')
        for m in models:
            print(f'   {m["id"]}')
    else:
        print(f'❌ vLLM error: {resp.status_code}')
except Exception as e:
    print(f'❌ vLLM unreachable: {e}')

# Quick chat test
print('\n── Chat test (Gemma) ──')
try:
    resp = requests.post(
        f'{VLLM_URL}/v1/chat/completions',
        json={
            'model': 'google/gemma-4-31B-it',
            'messages': [{'role': 'user', 'content': 'Say hello in one word.'}],
            'max_tokens': 50,
        },
        timeout=30,
    )
    if resp.status_code == 200:
        content = resp.json()['choices'][0]['message']['content']
        print(f'✅ Chat works — response: {content[:80]}')
    else:
        print(f'❌ Chat error: {resp.status_code} — {resp.text[:200]}')
except Exception as e:
    print(f'❌ Chat failed: {e}')

# Check Fireworks API
print('\n── Fireworks API (embeddings) ──')
try:
    resp = requests.get(
        'https://api.fireworks.ai/inference/v1/models',
        headers={'Authorization': f'Bearer {FIREWORKS_API_KEY}'},
        timeout=10,
    )
    if resp.status_code == 200:
        print('✅ Fireworks API reachable')
    else:
        print(f'❌ Fireworks error: {resp.status_code}')
except Exception as e:
    print(f'❌ Fireworks unreachable: {e}')

## 5. Configure run parameters

Edit these before running the benchmark.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  EDIT THESE PARAMETERS                                        ║
# ╚══════════════════════════════════════════════════════════════╝

TIER = '100K'              # Options: 100K, 500K, 1M
MAX_SAMPLES = 4            # Number of conversations (-1 = all)
MAX_QUESTIONS = -1         # Questions per type per chat (-1 = all)
CHUNK_SIZE = 3000          # Chunk size in chars (3000 recommended for 100K)
RETRIEVAL = 'hybrid'       # Options: hybrid, graph_completion, dense, lexical, chunk
MAX_CONTEXT_CHARS = ''     # Truncate conversation (empty = no truncation)

# ── Storage paths (GlusterFS home dir — persistent across sessions) ──
RESULTS_BASE = os.path.expanduser('~/beam_results')
KG_DIR = f'{RESULTS_BASE}/kg_cache/{TIER}'
CHECKPOINT_DIR = f'{RESULTS_BASE}/checkpoints/{TIER}'
RESPONSES_PATH = f'{RESULTS_BASE}/responses/beam_{TIER}_responses.json'
SCORES_PATH = f'{RESULTS_BASE}/scores/beam_{TIER}_scores.json'

# Create directories
!mkdir -p {KG_DIR} {CHECKPOINT_DIR} {RESULTS_BASE}/responses {RESULTS_BASE}/scores

# Use the beam env's Python for all benchmark commands
BEAM_PY = '/opt/conda/envs/beam/bin/python'

# Build the command
cmd = f"""
{BEAM_PY} evaluation/BEAM/run_eval.py \\
    --tier {TIER} \\
    --max-samples {MAX_SAMPLES} \\
    --max-questions {MAX_QUESTIONS} \\
    --chunk-size {CHUNK_SIZE} \\
    --retrieval {RETRIEVAL} \\
    --model google/gemma-4-31B-it \\
    --embedding-model accounts/fireworks/models/qwen3-embedding-8b \\
    --save-kg-dir {KG_DIR} \\
    --checkpoint-dir {CHECKPOINT_DIR} \\
    --resume \\
    --output {RESPONSES_PATH}
"""
if MAX_CONTEXT_CHARS:
    cmd += f"    --max-context-chars {MAX_CONTEXT_CHARS} \\\n"

print('Run command:')
print(cmd)
print(f'\nKG cache     → {KG_DIR}')
print(f'Checkpoints  → {CHECKPOINT_DIR}')
print(f'Responses    → {RESPONSES_PATH}')
print(f'Scores       → {SCORES_PATH}')

## 6. Run BEAM Phase 1 — Build KG + Answer Questions

**This is the long-running cell.** Keep the JupyterHub tab open.

With the local GPU cluster (no rate limits) and the batch size fix, this should take **~30-60 min per conversation** instead of 12+ hours.

If JupyterHub disconnects:
- Results saved so far are in `~/beam_results/` (GlusterFS, persistent)
- Checkpoints are in `~/beam_results/checkpoints/`
- Re-run this cell — `--resume` will skip completed stages

In [ ]:
import time, json, os, sys, subprocess, platform, datetime, csv, re, shutil
from pathlib import Path

# ═══════════════════════════════════════════════════════════════════
# 6. Run BEAM Phase 1 — Build KG + Answer Questions  (data-capturing)
# ═══════════════════════════════════════════════════════════════════
# Streams run output to a timestamped log file on GlusterFS and writes
# a comprehensive run archive (config, env, git SHA, KG stats, per-Q
# coverage/confidence/gaps/abstentions, errors) alongside responses.
# Post-processing runs in `finally` so the archive is produced even if
# the kernel is interrupted mid-run.

start_dt = datetime.datetime.now()
RUN_ID = f"beam_{TIER}_{start_dt.strftime('%Y%m%d_%H%M%S')}"
ARCHIVE_DIR = Path(RESULTS_BASE) / 'archives' / RUN_ID
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = ARCHIVE_DIR / 'phase1.log'
ARCHIVE_PATH = ARCHIVE_DIR / 'run_archive.json'
SUMMARY_PATH = ARCHIVE_DIR / 'summary.txt'
CSV_PATH = ARCHIVE_DIR / 'per_question.csv'

print(f'Starting BEAM Phase 1 at {start_dt.strftime("%H:%M:%S")}')
print(f'Tier: {TIER} | Samples: {MAX_SAMPLES} | Retrieval: {RETRIEVAL}')
print(f'Model: google/gemma-4-31B-it (gpu02 vLLM via SSH tunnel)')
print(f'Embeddings: qwen3-embedding-8b (Fireworks)')
print(f'Run ID:   {RUN_ID}')
print(f'Archive:  {ARCHIVE_DIR}')
print('=' * 60)


# ── helpers (defined BEFORE use) ───────────────────────────────────
def _snapshot_env():
    env = {
        'python_version': sys.version.split()[0],
        'python_executable': sys.executable,
        'hostname': platform.node(),
        'platform': platform.platform(),
    }
    try:
        env['git_sha'] = subprocess.check_output(
            ['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
        env['git_branch'] = subprocess.check_output(
            ['git', 'rev-parse', '--abbrev-ref', 'HEAD'], cwd=REPO_DIR, text=True).strip()
        env['git_dirty'] = bool(subprocess.check_output(
            ['git', 'status', '--porcelain'], cwd=REPO_DIR, text=True).strip())
        env['git_subject'] = subprocess.check_output(
            ['git', 'log', '-1', '--format=%s'], cwd=REPO_DIR, text=True).strip()
    except Exception as e:
        env['git_error'] = str(e)
    pkgs = {}
    for pkg in ('multi_agent_kg', 'datasets', 'httpx', 'openai', 'pydantic',
                'sentence_transformers', 'scipy', 'numpy', 'tiktoken'):
        try:
            mod = __import__(pkg)
            pkgs[pkg] = getattr(mod, '__version__', 'unknown')
        except Exception:
            pkgs[pkg] = 'not-installed'
    env['packages'] = pkgs
    return env


def _snapshot_config():
    cfg = {
        'tier': TIER, 'max_samples': MAX_SAMPLES, 'max_questions': MAX_QUESTIONS,
        'chunk_size': CHUNK_SIZE, 'retrieval': RETRIEVAL,
        'max_context_chars': MAX_CONTEXT_CHARS or None,
        'model': 'google/gemma-4-31B-it',
        'embedding_model': 'accounts/fireworks/models/qwen3-embedding-8b',
        'judge_model': os.environ.get('BEAM_JUDGE_MODEL', 'accounts/fireworks/models/deepseek-v4-pro'),
        'vllm_base_url': os.environ.get('VLLM_BASE_URL', 'http://127.0.0.1:8000/v1'),
        'entity_extract_workers': os.environ.get('ENTITY_EXTRACT_WORKERS'),
        'relation_extract_workers': os.environ.get('RELATION_EXTRACT_WORKERS'),
        'kg_dir': KG_DIR, 'checkpoint_dir': CHECKPOINT_DIR,
        'responses_path': RESPONSES_PATH, 'scores_path': SCORES_PATH,
    }
    try:
        r = requests.get(f'{VLLM_URL}/v1/models', timeout=10)
        if r.status_code == 200:
            cfg['vllm_models'] = [m.get('id') for m in r.json().get('data', [])]
    except Exception as e:
        cfg['vllm_models_error'] = str(e)
    try:
        env_txt = Path('.env').read_text()
        env_txt = re.sub(r'(KEY=)\S+', r'\1<REDACTED>', env_txt)
        cfg['env_file_redacted'] = env_txt
    except Exception:
        pass
    return cfg


def _parse_log(text):
    """Parse the Phase-1 log for per-chat + per-question telemetry + errors."""
    out = {'chats': [], 'errors': [], 'warnings': [],
           'per_question': [], 'failed_documents': 0}
    cur_chat = None
    cur_q = None
    chat_re = re.compile(r'INFO beam_eval:\s*=== Chat (\d+)/(\d+) — ID: (\S+) ===')
    build_re = re.compile(r'INFO beam_eval:\s+Built: (\d+) entities, (\d+) triples in ([\d.]+)s')
    q_re = re.compile(r'INFO beam_eval:\s+\[(\w+)\] Q(\d+): (.+?) → (.+?) \(([\d.]+)s\)')
    cov_re = re.compile(r'Overall coverage:\s*([\d.]+)')
    conf_re = re.compile(r'Overall confidence:\s*([\d.]+)')
    prov_re = re.compile(r'Provenance:\s*(\d+)/(\d+) grounded')
    gap_re = re.compile(r'Gaps:\s*(\[.*\])')
    fail_re = re.compile(r'Failed Documents:\s*(\d+)')
    for line in text.splitlines():
        m = chat_re.search(line)
        if m:
            cur_chat = {'idx': int(m.group(1)), 'total': int(m.group(2)),
                        'id': m.group(3), 'build_entities': None,
                        'build_triples': None, 'build_time_s': None}
            out['chats'].append(cur_chat)
            cur_q = None
            continue
        m = build_re.search(line)
        if m and cur_chat is not None:
            cur_chat['build_entities'] = int(m.group(1))
            cur_chat['build_triples'] = int(m.group(2))
            cur_chat['build_time_s'] = float(m.group(3))
            continue
        m = q_re.search(line)
        if m:
            cur_q = {'type': m.group(1), 'q_num': int(m.group(2)),
                     'question': m.group(3)[:120],
                     'response': m.group(4)[:120],
                     'duration_s': float(m.group(5)),
                     'chat_id': cur_chat['id'] if cur_chat else None}
            out['per_question'].append(cur_q)
            continue
        if cur_q is not None:
            m = cov_re.search(line)
            if m: cur_q['coverage'] = float(m.group(1)); continue
            m = conf_re.search(line)
            if m: cur_q['confidence'] = float(m.group(1)); continue
            m = prov_re.search(line)
            if m:
                cur_q['provenance_grounded'] = int(m.group(1))
                cur_q['provenance_total'] = int(m.group(2))
                continue
            m = gap_re.search(line)
            if m:
                try: cur_q['gaps'] = eval(m.group(1))
                except Exception: cur_q['gaps_raw'] = m.group(1)
                continue
        m = fail_re.search(line)
        if m: out['failed_documents'] = int(m.group(1))
        if ' ERROR ' in line or line.startswith('ERROR'):
            out['errors'].append(line.strip()[:300])
        elif ' WARNING ' in line or line.startswith('WARNING'):
            if 'httpx' not in line.lower() and 'unauthenticated' not in line.lower():
                out['warnings'].append(line.strip()[:300])
    return out


def _post_process(archive, pre_snapshot, start_dt, exit_code, log_path):
    """Load responses + KG stats + parse log → finalize archive."""
    end_dt = datetime.datetime.now()
    elapsed = (end_dt - start_dt).total_seconds()
    print(f'\n{"=" * 60}\nPost-processing run archive (elapsed {elapsed:.0f}s)...')

    archive['end_iso'] = end_dt.isoformat()
    archive['elapsed_s'] = round(elapsed, 1)
    archive['exit_code'] = exit_code
    archive['log_path'] = str(log_path)

    # Load responses JSON
    responses_data = None
    try:
        responses_data = json.loads(Path(RESPONSES_PATH).read_text())
        archive['responses_loaded'] = True
        archive['responses_meta'] = responses_data.get('meta', {})
    except Exception as e:
        archive['responses_loaded'] = False
        archive['responses_error'] = str(e)

    # Load KG stats from each saved KG (stats embedded in the JSON)
    kg_stats_by_chat = {}
    kg_dir_path = Path(KG_DIR)
    if kg_dir_path.exists():
        for kg_sub in sorted(kg_dir_path.iterdir()):
            if not kg_sub.is_dir():
                continue
            kg_json = kg_sub / 'knowledge_graph.json'
            if not kg_json.exists():
                candidates = list(kg_sub.glob('*.json'))
                kg_json = candidates[0] if candidates else None
            if kg_json and kg_json.exists():
                try:
                    kd = json.loads(kg_json.read_text())
                    kg_stats_by_chat[kg_sub.name] = {
                        'stats': kd.get('stats', {}),
                        'kg_file': str(kg_json),
                        'governance_mode': kd.get('governance_mode'),
                    }
                except Exception as e:
                    kg_stats_by_chat[kg_sub.name] = {'load_error': str(e)}
    archive['kg_stats_by_chat'] = kg_stats_by_chat

    # Parse log
    log_text = Path(log_path).read_text() if Path(log_path).exists() else ''
    archive['log_parsed'] = _parse_log(log_text)

    # Enrich chats with KG stats + abstention flag
    abstention_prefixes = (
        "i don't have enough information",
        "i don't have enough information in the knowledge graph",
        "i do not have enough information",
        "the knowledge graph does not contain",
        "no information in the knowledge graph",
    )
    chats_enriched = []
    if responses_data:
        for chat in responses_data.get('chats', []):
            cid = str(chat.get('conversation_id'))
            chat_kg = kg_stats_by_chat.get(cid, {})
            enriched = dict(chat)
            enriched['kg_stats'] = chat_kg.get('stats', {})
            enriched['kg_file'] = chat_kg.get('kg_file')
            resp = chat.get('responses', {})
            q_summary = {'total': 0, 'answered': 0, 'abstained': 0,
                         'by_type': {}, 'query_times': []}
            flat_qs = []
            for q_type, qs in resp.items():
                q_summary['by_type'][q_type] = {'total': len(qs),
                    'answered': 0, 'abstained': 0}
                for q in qs:
                    q_summary['total'] += 1
                    ans = (q.get('llm_response') or '').strip().lower()
                    is_abst = ans.startswith(abstention_prefixes) or len(ans) < 5
                    q2 = dict(q)
                    q2['question_type'] = q_type
                    q2['is_abstention'] = is_abst
                    if is_abst:
                        q_summary['abstained'] += 1
                        q_summary['by_type'][q_type]['abstained'] += 1
                    else:
                        q_summary['answered'] += 1
                        q_summary['by_type'][q_type]['answered'] += 1
                    if '_query_time_s' in q:
                        q_summary['query_times'].append(q['_query_time_s'])
                    flat_qs.append(q2)
            enriched['question_summary'] = q_summary
            enriched['_flat_questions'] = flat_qs
            chats_enriched.append(enriched)
    archive['chats'] = chats_enriched

    # Aggregate
    agg = {
        'total_chats': len(chats_enriched),
        'total_entities': sum(c.get('kg_stats', {}).get('entities', 0) for c in chats_enriched),
        'total_triples': sum(c.get('kg_stats', {}).get('triples', 0) for c in chats_enriched),
        'total_orphan_entities': sum(c.get('kg_stats', {}).get('orphan_entities', 0) for c in chats_enriched),
        'total_questions': sum(c.get('question_summary', {}).get('total', 0) for c in chats_enriched),
        'total_answered': sum(c.get('question_summary', {}).get('answered', 0) for c in chats_enriched),
        'total_abstained': sum(c.get('question_summary', {}).get('abstained', 0) for c in chats_enriched),
        'total_build_time_s': sum(c.get('_build_time_s', 0) for c in chats_enriched),
        'by_type': {},
    }
    for c in chats_enriched:
        for qt, s in c.get('question_summary', {}).get('by_type', {}).items():
            d = agg['by_type'].setdefault(qt, {'total': 0, 'answered': 0, 'abstained': 0})
            d['total'] += s['total']
            d['answered'] += s['answered']
            d['abstained'] += s['abstained']
    agg['abstention_rate'] = round(agg['total_abstained'] / max(agg['total_questions'], 1), 4)
    all_qt = [t for c in chats_enriched for t in c.get('question_summary', {}).get('query_times', [])]
    if all_qt:
        agg['avg_query_time_s'] = round(sum(all_qt) / len(all_qt), 2)
        agg['max_query_time_s'] = round(max(all_qt), 2)
    archive['aggregate'] = agg

    # Write archive JSON
    ARCHIVE_PATH.write_text(json.dumps(archive, indent=2, default=str))

    # Per-question CSV
    try:
        with open(CSV_PATH, 'w', newline='') as f:
            w = csv.writer(f)
            w.writerow(['chat_id', 'question_type', 'q_idx', 'question',
                        'llm_response_preview', 'is_abstention',
                        'query_time_s', 'rubric_items'])
            for c in chats_enriched:
                cid = c.get('conversation_id')
                for i, q in enumerate(c.get('_flat_questions', [])):
                    w.writerow([cid, q.get('question_type'), i,
                                (q.get('question') or '')[:200],
                                (q.get('llm_response') or '')[:200],
                                q.get('is_abstention'),
                                q.get('_query_time_s'),
                                len(q.get('rubric') or [])])
    except Exception as e:
        print(f'  CSV write failed: {e}')

    # Human-readable summary
    try:
        with open(SUMMARY_PATH, 'w') as f:
            f.write(f'BEAM Phase 1 Run: {RUN_ID}\n{"=" * 60}\n')
            f.write(f'Start:    {archive["start_iso"]}\n')
            f.write(f'End:      {archive["end_iso"]}\n')
            f.write(f'Elapsed:  {elapsed:.0f}s ({elapsed/3600:.2f}h)\n')
            f.write(f'Exit:     {exit_code}\n')
            f.write(f'Git:      {archive["environment"].get("git_sha","?")[:12]} '
                    f'{archive["environment"].get("git_subject","")}\n')
            f.write(f'Config:   tier={TIER} samples={MAX_SAMPLES} '
                    f'retrieval={RETRIEVAL} chunk={CHUNK_SIZE}\n')
            f.write(f'\n--- Aggregate ---\n')
            for k, v in agg.items():
                f.write(f'  {k}: {v}\n')
            f.write(f'\n--- Per-chat ---\n')
            for c in chats_enriched:
                ks = c.get('kg_stats', {})
                qs = c.get('question_summary', {})
                f.write(f'  chat {c.get("conversation_id")}: '
                        f'{ks.get("entities",0)} ent / {ks.get("triples",0)} trip / '
                        f'{ks.get("orphan_entities",0)} orphan / '
                        f'build {c.get("_build_time_s",0):.0f}s / '
                        f'{qs.get("total",0)} Q ({qs.get("answered",0)} ans, '
                        f'{qs.get("abstained",0)} abst)\n')
            lp = archive.get('log_parsed', {})
            if lp.get('errors'):
                f.write(f'\n--- Errors ({len(lp["errors"])}) ---\n')
                for er in lp['errors'][:20]:
                    f.write(f'  {er}\n')
            if lp.get('warnings'):
                f.write(f'\n--- Warnings ({len(lp["warnings"])}) ---\n')
                for wr in lp['warnings'][:20]:
                    f.write(f'  {wr}\n')
    except Exception as e:
        print(f'  Summary write failed: {e}')

    # Print summary
    print(f'\n{"=" * 60}')
    print(f'Phase 1 complete in {elapsed/3600:.1f} hours (exit {exit_code})')
    print(f'Responses:  {RESPONSES_PATH}')
    print(f'Archive:    {ARCHIVE_PATH}')
    print(f'Log:        {LOG_PATH}')
    print(f'Summary:    {SUMMARY_PATH}')
    print(f'CSV:        {CSV_PATH}')
    print(f'\n--- Aggregate ---')
    for k, v in agg.items():
        print(f'  {k}: {v}')
    print(f'\n--- Per-chat ---')
    for c in chats_enriched:
        ks = c.get('kg_stats', {})
        qs = c.get('question_summary', {})
        print(f'  chat {c.get("conversation_id")}: '
              f'{ks.get("entities",0)} ent / {ks.get("triples",0)} trip / '
              f'{ks.get("orphan_entities",0)} orphan / '
              f'build {c.get("_build_time_s",0):.0f}s / '
              f'{qs.get("total",0)} Q ({qs.get("answered",0)} ans, '
              f'{qs.get("abstained",0)} abst)')


# ── 1. Pre-run snapshot ────────────────────────────────────────────
pre_snapshot = {
    'run_id': RUN_ID,
    'start_iso': start_dt.isoformat(),
    'config': _snapshot_config(),
    'environment': _snapshot_env(),
    'phase1_command': cmd.strip(),
}
ARCHIVE_PATH.write_text(json.dumps(pre_snapshot, indent=2, default=str))
print(f'  Pre-run snapshot → {ARCHIVE_PATH}')

# ── 2. Run Phase 1 (stream to log; filter notebook output) ─────────
_PRINT_RE = re.compile(
    r'(INFO beam_eval|ERROR|WARNING|===|Built:|Failed Documents|'
    r'Phase 1 complete|Saved progress|Processing coref batch|'
    r'Segment \d+/(?:318|272)\b|DomainBuilder|DELIBERATIVE MULTI-AGENT)',
    re.IGNORECASE,
)
_SUPPRESS_RE = re.compile(r'INFO httpx:|Generating \d+K split')

proc = subprocess.Popen(
    cmd.strip(), shell=True,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
exit_code = None
try:
    with open(LOG_PATH, 'w') as logf:
        for line in proc.stdout:
            logf.write(line)
            logf.flush()
            if _SUPPRESS_RE.search(line):
                continue
            if _PRINT_RE.search(line):
                print(line.rstrip())
    exit_code = proc.wait()
    print(f'\n[Phase 1 exited with code {exit_code}]')
except KeyboardInterrupt:
    proc.terminate()
    try:
        proc.wait(timeout=30)
    except subprocess.TimeoutExpired:
        proc.kill()
    exit_code = proc.returncode
    print(f'\n[INTERRUPTED — exit code {exit_code}]')
finally:
    _post_process(dict(pre_snapshot), pre_snapshot, start_dt, exit_code, LOG_PATH)

## 7. Run BEAM Phase 2 — Score with LLM Judge

Uses the local Gemma model as judge (free, no rate limits). Takes ~1-3 hours for 4 samples.

If interrupted, re-run with `--resume` to skip already-scored questions.

In [ ]:
import time

BEAM_PY = '/opt/conda/envs/beam/bin/python'

start = time.time()
print(f'Starting BEAM Phase 2 (Scoring) at {time.strftime("%H:%M:%S")}')
print(f'Judge model: accounts/fireworks/models/deepseek-v4-pro (Fireworks)')
print('=' * 60)

!{BEAM_PY} evaluation/BEAM/score.py \
    --responses {RESPONSES_PATH} \
    --output {SCORES_PATH} \
    --judge-model accounts/fireworks/models/deepseek-v4-pro \
    --resume

elapsed = time.time() - start
print(f'\n{"=" * 60}')
print(f'Phase 2 complete in {elapsed/3600:.1f} hours')
print(f'Scores saved to: {SCORES_PATH}')

## 8. View aggregate results

In [ ]:
import json
from pathlib import Path

scores_path = Path(SCORES_PATH)
if scores_path.exists():
    with open(scores_path) as f:
        data = json.load(f)
    
    print('=' * 60)
    print('BEAM Aggregate Scores')
    print('=' * 60)
    print(json.dumps(data.get('aggregate', data), indent=2))
else:
    print(f'Scores file not found at {scores_path}')
    print('Check if Phase 2 completed successfully.')

## 9. Resume interrupted run

If JupyterHub disconnected mid-run, use this cell to resume.

- **Checkpoints** (`--checkpoint-dir + --resume`): Skips completed pipeline stages (entity extraction, relation extraction, evidence linking, etc.) within each conversation.
- **KG cache** (`--save-kg-dir`): Completed conversations' KGs are saved. Use `--load-kg-dir` to skip KG building entirely for those.

Both mechanisms work together: checkpoints resume mid-conversation, KG cache skips fully-built conversations.

In [ ]:
BEAM_PY = '/opt/conda/envs/beam/bin/python'

# Resume Phase 1 — continues from last completed checkpoint stage.
# If a conversation's KG was fully built and saved, it will be loaded
# from --save-kg-dir instead of rebuilt.
!{BEAM_PY} evaluation/BEAM/run_eval.py \
    --tier {TIER} \
    --max-samples {MAX_SAMPLES} \
    --max-questions {MAX_QUESTIONS} \
    --chunk-size {CHUNK_SIZE} \
    --retrieval {RETRIEVAL} \
    --model google/gemma-4-31B-it \
    --embedding-model accounts/fireworks/models/qwen3-embedding-8b \
    --save-kg-dir {KG_DIR} \
    --load-kg-dir {KG_DIR} \
    --checkpoint-dir {CHECKPOINT_DIR} \
    --resume \
    --output {RESPONSES_PATH}

print('\n=== Resume complete ===')

## 10. Check GPU status (optional)

Monitor GPU utilization on gpu02 while the benchmark runs.

In [ ]:
# Check GPU status and vLLM log from a JupyterHub terminal:
#
#   ssh gpu02.mind.cs.umd.edu 'nvidia-smi'
#   ssh gpu02.mind.cs.umd.edu 'tail -f /scratch/vllm_gemma.log'
#
# Or just poll vLLM's health endpoint from here:
import requests

try:
    resp = requests.get(f'{VLLM_URL}/v1/models', timeout=5)
    if resp.status_code == 200:
        models = resp.json().get('data', [])
        print(f'✅ vLLM alive — {len(models)} model(s): {[m["id"] for m in models]}')
    else:
        print(f'⚠️  vLLM returned {resp.status_code}')
except Exception as e:
    print(f'❌ vLLM unreachable: {e}')